# SUPP: Shared human/bovine peptides labelled as one species, per submission.

```
SUPP: Shared human/bovine peptides labelled as one species, per submission.

Explains the two human groups in FIG03B. Each participant's *-pepQuant peptides are
classified by SEQUENCE (in-silico tryptic digest of the study FASTA, same rules as
fig5_homology.ipynb), then compared with the species label the participant gave.

  Panel A  composition of the peptides each submission LABELLED "Human", by sequence class
           (human-only, human+bovine shared, all three, other).
  Panel B  human A/C per submission: median over all "Human"-labelled peptides (= FIG03B)
           vs. median over sequence-unique human peptides only; dashed = expected 15.

Denominator: rows the participant labelled Human with a finite A/C (precursor-level rows
for 07A/07B, as submitted). Same 15 files, curation, data-type handling and A/C rules as
FIG03/FIG04.

  raw {01,02,03,04,07A,07B,09,11,12,14A,14B}; log2 {06}; scaled {10A,10B};
  counts {41} (+1 pseudocount -> compressed ratio; flagged with *).

EXCLUDED: 05, 42, 14/Templates (placeholder data).
Fixups: site-10-QEx mislabeled Sample C col; site-09 per-replicate columns averaged.
Outputs: output/SUPP_shared_peptide_labels.{png,pdf};
         data/supp_shared_labels_long.csv, data/supp_shared_labels_summary.csv.
```

In [1]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

DATA_DIR = r"D:/2022 Multi-Species Standard Study"
FASTA = DATA_DIR + "/fasta/LakeTrout-Human-Cow-Contaminants-sPRG2022.fasta"
OUTPUT = "output"
DATA = "data"
os.makedirs(OUTPUT, exist_ok=True)
os.makedirs(DATA, exist_ok=True)

CURATED = {   # label -> pepQuant file (relative to DATA_DIR); same as FIG03/FIG04
    "01": "01/01-pepQuant.tsv", "02": "02/02-pepQuant.csv", "03": "03/03-pepQuant.tsv",
    "04": "04/04-pepQuant.tsv", "06": "06/06-pepQuant.tsv",
    "07A": "07/07-pepQuant.tsv", "07B": "07/07-pepQuant_ShortGrad.tsv",
    "09": "09/09-pepQuant.tsv",
    "10A": "10/10-QEx_pepQuant.tsv", "10B": "10/10-Fusion_pepQuant.tsv",
    "11": "11/11-pepQuant.tsv", "12": "12/12-pepQuant.tsv",
    "14A": "14/14_pepQuant_sPRG_lumos_DIA_1x8mzStag_3x4mzGPFlibrary.txt",
    "14B": "14/14_pepQuant_sPRG_lumos_DIA_2x4mzStag_Prosit_library.txt",
    "41": "41/41-pepQuant.tsv",
}
PEPTIDE_COLS = ("Peptide", "EG.PrecursorId", "Pepide", "Sequence")   # 09 header typo is real
DATATYPE = {k: "raw" for k in CURATED}
DATATYPE.update({"06": "log2", "10A": "scaled", "10B": "scaled", "41": "counts"})

SPECIES_MAP = {"cow": "Bovine", "bovin": "Bovine", "bovine": "Bovine", "bos taurus": "Bovine",
               "human": "Human", "homo sapiens": "Human",
               "trout": "Trout", "salvelinus namaycush": "Trout", "salnm": "Trout"}
COLORS = {"Human": "#E69F00", "Bovine": "#009E73", "Trout": "#56B4E9"}  # human orange, cow green, trout blue
ORGBIT = {"Human": 1, "Bovine": 2, "Trout": 4}
CAT = {0: "Not in FASTA digest", 1: "Human only", 2: "Bovine only", 4: "Trout only",
       3: "Human+Bovine", 5: "Human+Trout", 6: "Bovine+Trout", 7: "All three"}
STACK = ["Human only", "Human+Bovine", "All three", "Other"]
SCOLOR = {"Human only": COLORS["Human"], "Human+Bovine": "#CC79A7",
          "All three": "#8C8C8C", "Other": "#D9D9D9"}
EXPECTED_HUMAN_AC = 45 / 3

In [2]:
def clean_pep(p):
    """Bare sequence from any submitted format: _PEP_.2, [R].PEP.[V], PEP(UniMod:4), [mods]."""
    s = re.sub(r"\.\d+$", "", str(p))      # charge suffix
    s = re.sub(r"\[.*?\]", "", s)           # bracketed mods and flanking residues
    s = re.sub(r"\(.*?\)", "", s)           # (UniMod:n)
    return re.sub(r"[^A-Z]", "", s.upper())

In [3]:
def digest(seq, missed=2, min_len=6, max_len=50):
    seq = re.sub(r"[^A-Z]", "", seq.upper())
    cuts = [0] + [i + 1 for i in range(len(seq) - 1) if seq[i] in "KR" and seq[i + 1] != "P"] + [len(seq)]
    cuts = sorted(set(cuts))
    peps = set()
    for i in range(len(cuts) - 1):
        for m in range(missed + 1):
            j = i + 1 + m
            if j >= len(cuts):
                break
            p = seq[cuts[i]:cuts[j]]
            if min_len <= len(p) <= max_len:
                peps.add(p)
    return peps


def fasta_org(header):
    tok = header.split()[0]
    if "Cont_" in tok:
        return None          # contaminant -- not part of the 3-proteome design
    if "_HUMAN" in tok:
        return "Human"
    if "_BOVIN" in tok:
        return "Bovine"
    return "Trout"           # trout is UniProt (_SALNM) in the study FASTA


def classify(targets):
    """peptide -> homology class, from which organisms' digests contain it."""
    member = {p: 0 for p in targets}

    def flush(header, parts):
        if not header:
            return
        org = fasta_org(header)
        if org is None:
            return
        for p in digest("".join(parts)):
            if p in member:
                member[p] |= ORGBIT[org]

    header, parts = None, []
    with open(FASTA, encoding="utf-8", errors="ignore") as fh:
        for line in fh:
            if line.startswith(">"):
                flush(header, parts)
                header, parts = line[1:].strip(), []
            else:
                parts.append(line.strip())
        flush(header, parts)
    return {p: CAT.get(b, "Other") for p, b in member.items()}

In [4]:
def _read(path):
    sep = "," if path.lower().endswith(".csv") else "\t"
    return pd.read_csv(path, sep=sep, engine="python", on_bad_lines="skip")


def load_peptides():
    """Per-row tidy frame: label, peptide, participant species label, log2(A/C)."""
    frames = []
    for label, pq in CURATED.items():
        df = _read(os.path.join(DATA_DIR, pq))
        df.columns = [str(c).strip() for c in df.columns]
        # site 10-QEx: 3rd quantity col mislabeled "Sample B Quantity" (dup) = Sample C
        if "Sample B Quantity.1" in df.columns and "Sample C Quantity" not in df.columns:
            df = df.rename(columns={"Sample B Quantity.1": "Sample C Quantity"})
        pcol = next(c for c in PEPTIDE_COLS if c in df.columns)
        scol = "Species" if "Species" in df.columns else "PG.Organisms"
        q = {}
        for L in ("A", "C"):
            cols = [c for c in df.columns if re.search(rf"Sample {L}(_R\d+)? Quantity", c)]
            q[L] = df[cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)
        dt = DATATYPE[label]
        if dt in ("raw", "scaled"):
            keep = (q["A"] > 0) & (q["C"] > 0); y = np.log2((q["A"] / q["C"]).where(keep))
        elif dt == "log2":
            keep = q["A"].notna() & q["C"].notna(); y = (q["A"] - q["C"]).where(keep)
        elif dt == "counts":
            keep = q["A"].notna() & q["C"].notna()
            y = np.log2((q["A"].where(keep) + 1) / (q["C"].where(keep) + 1))
        sub = pd.DataFrame({"label": label, "datatype": dt,
                            "peptide": df[pcol].map(clean_pep),
                            "species_label": df[scol].map(lambda x: SPECIES_MAP.get(str(x).strip().lower())),
                            "log2_AC": y})
        n_in = len(sub)
        sub = sub.dropna(subset=["species_label", "log2_AC"])
        sub = sub[np.isfinite(sub["log2_AC"]) & (sub["peptide"].str.len() > 0)]
        print(f"  {label:4} rows in={n_in:6d}  kept={len(sub):6d}  (peptide col: {pcol})")
        frames.append(sub)
    return pd.concat(frames, ignore_index=True)

In [5]:
def summarize(p):
    rows = []
    for label in CURATED:
        for sp in ("Human", "Bovine"):
            s = p[(p["label"] == label) & (p["species_label"] == sp)]
            uniq = s["seq_class"] == f"{sp} only"
            shared = s["seq_class"] == "Human+Bovine"
            rows.append(dict(label=label, datatype=DATATYPE[label], species_label=sp, n=len(s),
                             frac_unique=uniq.mean(), frac_shared_HB=shared.mean(),
                             frac_all_three=(s["seq_class"] == "All three").mean(),
                             ratio_all=2 ** s["log2_AC"].median(),
                             ratio_unique=2 ** s.loc[uniq, "log2_AC"].median(),
                             ratio_shared=2 ** s.loc[shared, "log2_AC"].median()))
    return pd.DataFrame(rows)

In [6]:
def figure(p, summ, out_png):
    h = summ[summ["species_label"] == "Human"].sort_values("ratio_all", ascending=False)
    order = list(h["label"])
    ypos = np.arange(len(order))
    names = [f"{l} *" if DATATYPE[l] == "counts" else l for l in order]

    hp = p[p["species_label"] == "Human"].copy()
    hp["stack"] = hp["seq_class"].where(hp["seq_class"].isin(STACK[:-1]), "Other")
    frac = (hp.groupby("label")["stack"].value_counts(normalize=True)
              .unstack(fill_value=0).reindex(index=order, columns=STACK, fill_value=0))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 6.2), sharey=True,
                                   gridspec_kw={"width_ratios": [1.15, 1], "wspace": 0.08})
    left = np.zeros(len(order))
    for cls in STACK:
        ax1.barh(ypos, frac[cls].values, left=left, color=SCOLOR[cls], edgecolor="white",
                 linewidth=0.5, height=0.75, label=cls)
        left += frac[cls].values
    ax1.set_yticks(ypos, names)
    ax1.invert_yaxis()
    ax1.set_xlim(0, 1)
    ax1.set_xlabel('fraction of peptides labelled "Human" by participant')
    ax1.set_ylabel("submission")
    ax1.set_title('A   Sequence class of "Human" peptides', loc="left", fontsize=11)
    ax1.legend(loc="upper center", bbox_to_anchor=(0.5, -0.11), ncol=4, frameon=False,
               fontsize=9, title="sequence class (in-silico digest)", title_fontsize=9)

    ax2.axvline(EXPECTED_HUMAN_AC, ls="--", lw=1.2, color=COLORS["Human"], zorder=1)
    for y, (_, r) in zip(ypos, h.iterrows()):
        ax2.plot([r["ratio_all"], r["ratio_unique"]], [y, y], color="0.6", lw=1, zorder=2)
    ax2.scatter(h["ratio_all"], ypos, s=46, color="black", zorder=3)
    ax2.scatter(h["ratio_unique"], ypos, s=46, facecolor="white",
                edgecolor=COLORS["Human"], linewidth=1.8, zorder=4)
    ax2.set_xscale("log")
    ax2.set_xlim(0.8, 30)
    ax2.set_xticks([1, 2, 5, 10, 15, 20], ["1", "2", "5", "10", "15", "20"])
    ax2.set_xlabel("human A/C ratio (median of per-peptide ratios, log scale)")
    ax2.set_title("B   Human A/C: all vs. unique peptides", loc="left", fontsize=11)
    ax2.grid(True, axis="x", lw=0.3)
    handles = [Line2D([0], [0], marker="o", ls="", color="black", label='all "Human" peptides (= FIG03B)'),
               Line2D([0], [0], marker="o", ls="", markerfacecolor="white",
                      markeredgecolor=COLORS["Human"], markeredgewidth=1.8, label="sequence-unique human only"),
               Line2D([0], [0], ls="--", color=COLORS["Human"], label="expected (15)")]
    ax2.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.11), ncol=2,
               frameon=False, fontsize=9)

    fig.suptitle("SUPP  Shared human/bovine peptides labelled as human explain the low human ratios"
                 "\n(* = count data, +1 pseudocount)", fontsize=12)
    fig.subplots_adjust(left=0.08, right=0.98, top=0.88, bottom=0.24)
    fig.savefig(out_png, dpi=300)
    fig.savefig(out_png.replace(".png", ".pdf"))
    plt.close(fig)
    print(f"  wrote {out_png}")

In [7]:
# ---- generate figure ----
p = load_peptides()
print(f"{len(p)} rows, {p['label'].nunique()} submissions; dtypes: {dict(p.dtypes.astype(str))}")
cls = classify(set(p["peptide"]))
p["seq_class"] = p["peptide"].map(cls)
print("\nrows by sequence class:")
print(p["seq_class"].value_counts().to_string())
summ = summarize(p)
p.to_csv(os.path.join(DATA, "supp_shared_labels_long.csv"), index=False, lineterminator="\n")
summ.to_csv(os.path.join(DATA, "supp_shared_labels_summary.csv"), index=False, lineterminator="\n")
with pd.option_context("display.width", 160, "display.float_format", "{:.3f}".format):
    print("\nexpected A/C: Human 15.0, Bovine 0.106")
    print(summ.to_string(index=False))
figure(p, summ, os.path.join(OUTPUT, "SUPP_shared_peptide_labels.png"))

  01   rows in=  6109  kept=  4603  (peptide col: Peptide)
  02   rows in= 18729  kept= 12920  (peptide col: Peptide)
  03   rows in=  3651  kept=  3239  (peptide col: Peptide)


  04   rows in= 13680  kept= 10413  (peptide col: Peptide)
  06   rows in=  6590  kept=  3727  (peptide col: Peptide)


  07A  rows in= 25442  kept= 18467  (peptide col: EG.PrecursorId)


  07B  rows in= 25208  kept= 18433  (peptide col: EG.PrecursorId)
  09   rows in=  7699  kept=  5409  (peptide col: Pepide)
  10A  rows in=  7945  kept=  4881  (peptide col: Peptide)


  10B  rows in=  9063  kept=  6202  (peptide col: Peptide)
  11   rows in=  3757  kept=  3757  (peptide col: Peptide)
  12   rows in= 14662  kept= 14662  (peptide col: Peptide)


  14A  rows in= 14037  kept=  9523  (peptide col: Sequence)
  14B  rows in= 17361  kept= 11511  (peptide col: Sequence)
  41   rows in=  9852  kept=  9848  (peptide col: Peptide)
137595 rows, 15 submissions; dtypes: {'label': 'str', 'datatype': 'str', 'peptide': 'str', 'species_label': 'str', 'log2_AC': 'float64'}



rows by sequence class:
seq_class
Bovine only            40973
Trout only             35211
Human only             31770
Human+Bovine           19543
All three               7560
Not in FASTA digest     1690
Human+Trout              460
Bovine+Trout             388



expected A/C: Human 15.0, Bovine 0.106
label datatype species_label    n  frac_unique  frac_shared_HB  frac_all_three  ratio_all  ratio_unique  ratio_shared
   01      raw         Human 1618        0.980           0.000           0.000     10.413        10.432           NaN
   01      raw        Bovine 1886        0.972           0.000           0.000      0.123         0.123           NaN
   02      raw         Human 4925        0.344           0.436           0.192      1.577        12.212         1.229
   02      raw        Bovine 4818        0.528           0.326           0.106      0.257         0.152         0.815
   03      raw         Human  878        0.931           0.069           0.000     14.075        15.477         1.057
   03      raw        Bovine 1209        1.000           0.000           0.000      0.106         0.106           NaN
   04      raw         Human 4802        0.334           0.425           0.206      1.436         9.302         1.070
   04      raw  

  wrote output\SUPP_shared_peptide_labels.png
